# 4. PCAを用いた1段階目の次元削減（1024 → N）

## 4.1 ライブラリのインポート

In [11]:
import os
import numpy as np
from sklearn.decomposition import PCA
from tqdm import tqdm
import re
import bisect

## 4.2 変数の設定

In [ ]:
model_name = "prott5"

pca_components = 1024 # 圧縮する次元はデータにより異なるため調整の必要はない
batch_rows = 50000  # partial_fit に投入する 行数 の目安（調整可）
border = 0.8 # 累積寄与率

## 4.3 PCAの実行

In [ ]:
for fold in range(1, 6):
    data_dir = f"./data/embedding-vectors_exclusion/{model_name}/base/fold{fold}_train"  # 1024次元の埋め込みベクトルの読み込み
    files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".npy")]  # .npy ファイルを全て取得
    sorted_files = sorted(files, key=lambda p: int(re.search(r'(\d+).npy$', p).group(1)))
    # print(sorted_files) #出力が長いため非表示
    
    test_dir = f"./data/embedding-vectors_exclusion/{model_name}/base/fold{fold}_test"  # 1024次元の埋め込みベクトルの読み込み
    test_files = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith(".npy")]  # .npy ファイルを全て取得
    test_files = sorted(test_files, key=lambda p: int(re.search(r'(\d+).npy$', p).group(1)))
    # print(test_files) #出力が長いため非表示

    arr_cog = np.load(f"./data/embedding-vectors_exclusion/{model_name}/cog1024/fold{fold}_cog.npy", mmap_mode="r") #重心ベクトルを取得
    
    pca = PCA(n_components=pca_components)
    chunk = arr_cog.astype(np.float32)
    pca.fit(chunk)
    variance_ratio = pca.explained_variance_ratio_.cumsum().tolist()
    dim = bisect.bisect(variance_ratio, border)
    print("dim:", dim)

    print(pca.explained_variance_ratio_.cumsum()[0:dim+1])

    output_dir = f"./data/embedding-vectors_exclusion/{model_name}/fold_pca_cog/fold{fold}_train_pca_cog"
    os.makedirs(output_dir, exist_ok=True)
    output_dir_test = f"./data/embedding-vectors_exclusion/{model_name}/fold_pca_cog/fold{fold}_test_pca_cog"
    os.makedirs(output_dir_test, exist_ok=True)

    for filename in tqdm(sorted_files):
        arr = np.load(filename, mmap_mode="r").astype(np.float32)
        out = pca.transform(arr)
        out_dim = out[0:, 0:dim + 1]
        base = os.path.basename(filename).replace(".npy", "")
        np.save(os.path.join(output_dir, base + f"_pca.npy"), out_dim)
    
    for filename in tqdm(test_files):
        arr = np.load(filename, mmap_mode="r").astype(np.float32)
        out = pca.transform(arr)
        out_dim = out[0:, 0:dim + 1]
        base = os.path.basename(filename).replace(".npy", "")
        np.save(os.path.join(output_dir_test, base + f"_pca.npy"), out_dim)

['./data/embedding-vectors_exclusion/protbert/base/fold1_train\\1.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\2.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\4.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\5.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\6.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\7.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\8.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\9.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\10.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\13.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\14.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\17.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\18.npy', './data/embedding-vectors_exclusion/protbert/base/fold1_train\\19.npy',

100%|██████████| 1523/1523 [00:14<00:00, 103.88it/s]


['./data/embedding-vectors_exclusion/protbert/base/fold2_train\\1.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\3.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\4.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\5.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\6.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\7.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\8.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\9.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\11.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\12.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\13.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\15.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\16.npy', './data/embedding-vectors_exclusion/protbert/base/fold2_train\\17.npy',

 14%|█▍        | 880/6089 [00:08<00:51, 101.32it/s]


KeyboardInterrupt: 

## 4.4 各次元数における累積寄与率の確認

In [ ]:
print(pca.explained_variance_ratio_.cumsum()[0:dim])
a = np.load("./data/embedding-vectors_exclusion/prott5/base/1.npy")
print(len(a[0]))

[0.28785393 0.4511971  0.5603143  0.6067248  0.64519566 0.67604643
 0.7009696  0.7218862  0.7406919  0.7580588  0.7693587  0.7795674
 0.78938687 0.7986766  0.80680364]
1024
